# Apache Parquet Explained: A Guide for Data Professionals

This notebook transcribes the content from the PDF tutorial into markdown and Python cells.

- Date: February 2025
- Read time: 13 min read
- Author: Laiba Siddiqui
- Editor: Thalia Barrera

Big data can be overwhelming - we often feel there is no easy way to manage it. But in reality, Apache Parquet makes this so much easier. It is a smart data storage format that handles large datasets by saving you time and resources.

In this article, I will walk you through what makes Parquet a unique tool and how to use it in your projects. By the end, you will know why it is a top choice for data professionals and how to start using it with tools like Python and Spark.

## What is Apache Parquet?

Apache Parquet is an open-source columnar storage format that addresses big data processing challenges. Unlike traditional row-based storage, it organizes data into columns. This structure allows you to read only the necessary columns, making data queries faster and reducing resource consumption.

For example, you can target the relevant data instead of processing an entire dataset to find a single attribute. That is a primary reason why Parquet is a natural fit for big data frameworks like Apache Hadoop, Apache Spark, and Apache Hive.

Apart from frameworks, Parquet is also widely used in data lakes and analytics platforms. Teams use Amazon S3, Azure Data Lake Storage, or Google Cloud Storage to store large-scale datasets in a data lake. Since Parquet is optimized for efficient querying, it is a preferred format for storing structured and semi-structured data.

For example, in Amazon S3, a typical workflow might involve using AWS Glue to catalog Parquet files and Amazon Athena to run SQL queries without loading data into a database.

## Features of Apache Parquet

### Columnar storage
Unlike row-based formats like CSV, Parquet organizes data in columns. This means when we run a query, it only pulls the specific columns we need instead of loading everything. This improves performance and reduces I/O usage.

Parquet files are split into row groups, which hold a batch of rows. Each row group is broken into column chunks, each containing data for one column. These chunks are further divided into smaller pieces called pages, which are compressed to save space.

In addition, Parquet files store extra information in the footer, called metadata, which locates and reads only the data we need.

Original PDF image references were omitted, but the file structure discussed is: row groups -> column chunks -> pages -> footer metadata.

### Row groups
A row group contains multiple rows but stores data column-wise for efficient reading. Example: a dataset with 1 million rows might be split into 10 groups of 100,000 rows each.

### Column chunks
Within each row group, data is separated by columns. This design allows columnar pruning, where we can read only the relevant columns instead of scanning the entire file.

### Pages
Each column chunk is further split into pages to optimize memory usage. Pages are typically compressed, reducing storage costs.

### Footer (metadata)
The footer at the end of a Parquet file stores index information:
- Schema: Defines data types and column names.
- Row group offsets: Helps locate specific data quickly.
- Statistics: Min/max values to enable predicate pushdown (filtering at the storage level).

### Compression and encoding
Parquet compresses data column by column using compression methods like Snappy and Gzip. It also uses two encoding techniques:
- Run-length encoding to store repeated values compactly.
- Dictionary encoding to replace duplicates with dictionary references.

This reduces file sizes and speeds up data reading, which is especially helpful when you work with big data.

### Schema evolution
Schema evolution means modifying the structure of datasets, such as adding or altering columns. CSV is a simple text-based format with no built-in schema support, so any structural change requires rewriting the entire file. Older systems reading the modified file might also break if they expect a different structure.

With Parquet, you can add, remove, or update fields without breaking your existing files. Parquet stores schema information inside the file footer, allowing for evolving schemas without modifying existing files.

How this works in practice:
- When you add a new column, existing Parquet files remain unchanged.
- New files include the additional column, while old files still follow the previous schema.
- Removing a column does not require reprocessing previous data; queries ignore the missing column.
- If a column does not exist in an older file, Parquet engines like Spark, Hive, or BigQuery return NULL instead of breaking the query.
- Older Parquet files can be read even after schema modifications.
- Newer Parquet files with additional columns can still be read by systems expecting an older schema.

### Language and platform support
Parquet supports Java, Python, C++, and Rust. It is also natively integrated with Apache Spark, Hive, Presto, Flink, and Trino, ensuring efficient data processing at scale. Whether you are using Python through PySpark or another language, Parquet manages data in a way that makes it easy to query and analyze across different platforms.

## How to Read and Write Parquet Files

Now that you know the basics of Apache Parquet, the following cells walk through writing, reading, and integrating Parquet files with pandas, PyArrow, and big data frameworks like Spark.

In [3]:
# Install dependencies before working with Parquet files
# pip install pandas pyarrow

### Write Parquet files using pandas

In [4]:
import pandas as pd

# Sample DataFrame
data = {
    "Name": ["Alice", "Bob", "Charlie"],
    "Age": [25, 30, 35],
    "City": ["New York", "Los Angeles", "Chicago"]
}

df = pd.DataFrame(data)

# Write to Parquet file
df.to_parquet("data.parquet", engine="pyarrow", index=False)
print("Parquet file written successfully!")

Parquet file written successfully!


### Read Parquet files using pandas

In [5]:
import pandas as pd

# Read the Parquet file
df = pd.read_parquet("data.parquet", engine="pyarrow")
print("Data from Parquet file:")
print(df)

Data from Parquet file:
      Name  Age         City
0    Alice   25     New York
1      Bob   30  Los Angeles
2  Charlie   35      Chicago


### Write Parquet files using PyArrow

PyArrow is a tool from the Apache Arrow project that makes it easy to work with Parquet files.

In [6]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Sample data
df = pd.DataFrame({
    "Name": ["Jacob", "Lauren", "Oliver"],
    "Age": [25, 30, 35],
    "City": ["New York", "Los Angeles", "Chicago"]
})

# Convert to a PyArrow table
table = pa.Table.from_pandas(df)

# Write to Parquet file
pq.write_table(table, "data.parquet")
print("Parquet file written successfully!")

Parquet file written successfully!


### Read Parquet files using PyArrow

In [7]:
import pyarrow.parquet as pq

# Read the Parquet file
table = pq.read_table("data.parquet")

# Convert to a pandas DataFrame
df = table.to_pandas()
print("Data from Parquet file:")
print(df)

Data from Parquet file:
     Name  Age         City
0   Jacob   25     New York
1  Lauren   30  Los Angeles
2  Oliver   35      Chicago


### Integrate with big data frameworks

We can use Spark to read and write Parquet files directly. Download it from Apache Spark's website or set it up following the instructions, then import the libraries and create a DataFrame.

In [ ]:
import os
from pyspark.sql import SparkSession

os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SparkExample")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.faulthandler.enabled", "true")
    .getOrCreate()
)

# Define the schema for the dataset
schema = ["Name", "Age", "City"]

# Create a sample data
data = [
    ("Jacob", 30, "New York"),
    ("Lauren", 35, "Los Angeles"),
    ("Billy", 25, "Chicago")
]

# Create a DataFrame from the sample data
df = spark.createDataFrame(data, schema)

# minimal smoke test
spark.range(1).show()

# Show the DataFrame
df.show()

# Write DataFrame to Parquet
df.write.parquet("data.parquet")

In [ ]:
# Read the Parquet file
parquet_df = spark.read.parquet("data.parquet")

# Show the DataFrame
parquet_df.show()

+------+---+-----------+
|  Name|Age|       City|
+------+---+-----------+
| Jacob| 25|   New York|
|Lauren| 30|Los Angeles|
|Oliver| 35|    Chicago|
+------+---+-----------+



Apart from Spark, Parquet can also work with Hive. When you create a Hive table, use `STORED AS PARQUET` to make Parquet the storage format.

## Useful Operations with Parquet

Apart from reading and writing, there are some basic operations every developer should know since they are useful when working with Parquet files. The examples below use pandas and PyArrow to illustrate the concepts.

### Append data to an existing Parquet file
Appending data is useful when new records need to be added without rewriting the entire dataset.

In [9]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Load existing Parquet file
existing_table = pq.read_table("data.parquet")

# New data
new_data = pd.DataFrame({
    "Name": ["David", "Emma"],
    "Age": [40, 28],
    "City": ["San Francisco", "Seattle"]
})

# Convert new data to PyArrow table
new_table = pa.Table.from_pandas(new_data)

# Concatenate both tables
merged_table = pa.concat_tables([existing_table, new_table])

# Write back to Parquet file
pq.write_table(merged_table, "data.parquet")

### Read only specific columns from a Parquet file
Instead of loading the entire dataset, you can select only the necessary columns, reducing memory usage and improving performance. This is significantly faster than reading the full dataset.

In [10]:
df = pd.read_parquet("data.parquet", columns=["Name", "Age"])
print(df)

     Name  Age
0   Jacob   25
1  Lauren   30
2  Oliver   35
3   David   40
4    Emma   28


### Filtering data while reading (predicate pushdown)
Parquet allows efficient filtering at the storage level, known as predicate pushdown, which prevents loading unnecessary data. This avoids scanning the entire file, making queries much faster.

In [11]:
import pyarrow.parquet as pq

# Read only rows where Age > 30
table = pq.read_table("data.parquet", filters=[("Age", ">", 30)])
df = table.to_pandas()
print(df)

     Name  Age           City
0  Oliver   35        Chicago
1   David   40  San Francisco


### Merge multiple Parquet files
Often, Parquet files are stored as separate partitions. You can merge them into a single Parquet file. This is useful when combining datasets from different sources.

In [12]:
import pyarrow as pa
import pyarrow.parquet as pq

# List of Parquet files to merge
file_list = ["data.parquet", "data-2.parquet"]

# Read all files and merge
tables = [pq.read_table(f) for f in file_list]
merged_table = pa.concat_tables(tables)

# Write merged Parquet file
pq.write_table(merged_table, "merged_data.parquet")

### Convert CSV to Parquet
If you have existing CSV files, converting them to Parquet saves space and speeds up processing, which drastically reduces file size and improves read performance.

In [15]:
df = pd.read_csv("data.csv")
df.to_parquet("data.parquet", engine="pyarrow", index=False)

### Partition Parquet files for faster queries
Partitioning organizes data into subdirectories based on a column value, making queries significantly faster.

The original tutorial notes that this creates subdirectories such as:
- `partitioned_data/City=New York/`
- `partitioned_data/City=Los Angeles/`
- `partitioned_data/City=Chicago/`

You can then read only a specific partition. This speeds up analysis by scanning only relevant partitions.

In [16]:
df.to_parquet("partitioned_data/", engine="pyarrow", partition_cols=["City"])

df = pd.read_parquet("partitioned_data/City=New York/")
print(df)

     Name  Age
0  Oliver   35


### Use compression to optimize storage
Parquet supports compression algorithms like Snappy, Gzip, and Brotli to reduce file size.

In [17]:
df.to_parquet("compressed.parquet", engine="pyarrow", compression="snappy")

## Best Practices for Using Apache Parquet

When the author first started using Apache Parquet, minor adjustments greatly improved its efficiency. Top tips for optimizing Parquet in real-world scenarios:

### Choose the right compression codec
If you want to save storage, codecs like Snappy or Gzip can be good options. Snappy offers quick compression and decompression, which is useful when speed matters most. Gzip is ideal if you are tight on storage but can handle slightly slower reads. The key is understanding your workload. A faster codec like Snappy often wins when you frequently access files, while Gzip is best for archival data.

### Partition data effectively
Break your data into logical subsets, like dividing it by date, region, or any other frequently queried field to reduce the amount of data scanned during a query. The tutorial gives an example of partitioning years of transaction logs by year and month to fetch specific periods in seconds rather than minutes.

### Monitor schema evolution
Always ensure that new columns are added in a way that does not disrupt existing processes. This usually means appending them rather than modifying existing ones. Apache Spark's schema evolution support can help with smoother transitions.

## Apache Parquet vs Other Data Formats

### Parquet vs CSV
Parquet organizes data in columns, while CSV arranges it in rows. When you use Parquet, all the data from the same column are grouped together, so you can easily pull data from specific columns without sifting through everything else. It is faster and takes up less space because Parquet compresses data.

CSV stores data row by row. It is simple and works well for small datasets, but it is not ideal for big ones. Every query has to read the entire row, even if you only need a couple of columns. This slows things down and takes more memory to process.

### Parquet vs JSON
JSON is easy to understand, but it is not very efficient for storage or speed. JSON repeats column names and repeated values for every record, which makes files larger and slower. Parquet stores data by columns and compresses repeating values.

### Parquet vs Avro
Avro is a row-based format. It is useful for tasks like streaming data or processing logs, where you constantly add new records or retrieve complete rows. Parquet's column-based format is better for analytics because it reads the necessary columns and skips the rest to save time and resources. In short, Parquet is better for reading and analyzing large datasets, while Avro is ideal for writing and storing data in an easy-to-update way.

In [18]:
# JSON representation of the employee example from the tutorial
employee_data_json = [
    {"EmployeeID": 1, "Department": "HR", "Location": "New York"},
    {"EmployeeID": 2, "Department": "HR", "Location": "New York"},
    {"EmployeeID": 3, "Department": "HR", "Location": "New York"},
    {"EmployeeID": 4, "Department": "IT", "Location": "San Francisco"},
    {"EmployeeID": 5, "Department": "IT", "Location": "San Francisco"}
]

# The tutorial's conceptual Parquet layout for the same data
employee_id_column = [1, 2, 3, 4, 5]
department_column = ["HR", "HR", "HR", "IT", "IT"]
location_column = ["New York", "New York", "New York", "San Francisco", "San Francisco"]

### Comparison table

| Format | Pros | Cons | Use cases |
| --- | --- | --- | --- |
| Parquet | Columnar format for fast analytics; high compression efficiency; supports schema evolution; optimized for Spark, Hive, and Presto; supports predicate pushdown | Not human-readable; slower for row-based operations; more complex write operations | Big data processing, cloud data lakes, fast analytics, and OLAP workloads |
| CSV | Human-readable and simple; easy to generate and parse; compatible with almost all tools | No schema support; slow for large datasets; large file sizes; must scan entire file for queries | Data exchange, small to medium datasets, legacy interoperability |
| JSON | Supports nested and semi-structured data; human-readable; widely used in web APIs; flexible schema | Larger file sizes; slow for big data queries; no native indexing | Web APIs, NoSQL databases, streaming applications, logs, and semi-structured data |
| Avro | Row-based format for fast writes; compact binary format; supports schema evolution; good for streaming and message queues | Not human-readable; less efficient for analytics than Parquet; requires Avro libraries | Streaming and real-time processing, long-term storage with schema changes, efficient data serialization |

## When to Use Apache Parquet

Parquet is the best choice in situations like these:
- Analytics-heavy workloads: Parquet's columnar format lets you fetch only the data you need, which speeds up queries and saves time.
- Data lake architectures: Parquet's compression capabilities reduce stored data size without sacrificing performance.
- Large datasets: Parquet handles large and complex datasets neatly, especially those with nested or hierarchical structures, and its schema evolution support helps data models adapt over time.

## Final Thoughts

Apache Parquet is perfect for handling big data. It is fast, saves storage space, and works with tools like Spark.

Suggested follow-up resources from the tutorial:
- Introduction to Databricks course for understanding Databricks, a unified data platform to streamline big data workflows.
- Cleaning Data with PySpark course for cleaning and preprocessing datasets using PySpark.
- Big Data with PySpark track for scaling data processing with Apache Spark using the PySpark API.